In [79]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import firmware_sim as sim
import host_mapper as mapper

csv_path = '../data/measured_closed_loops/'
csv_file_name = 'closed_loop_18-08-2026_21-05-53_3_CCW_18m.csv'
df = pd.read_csv(f'{csv_path}{csv_file_name}', skipinitialspace=True)

In [80]:
state = sim.SystemState()

trajectory_x = []
trajectory_y = []

step_coords = []
step_coords.append(np.array([0.0, 0.0])) # Record the starting point

t_last = df['t_us'].iloc[0]

print("Starting firmware simulation test: ")

for index, row in df.iterrows():
    ax, ay, az = row['ax'], row['ay'], row['az']
    gx, gy, gz = row['gx'], row['gy'], row['gz']

    t_current = row['t_us']
    dt = (t_current - t_last) / 1e6
    t_last = t_current

    # first iteration will be < 0 probably
    if dt <= 0:
        dt = 0.005 # 200Hz

    is_zvw = sim.update_zvw(state.zvw, ax, ay, az, gx, gy, gz)
    # use instant quiet to keep fixing gyro during zvw
    sim.update_mahony(state.mahony, ax, ay, az, gx, gy, gz, dt, state.zvw.instant_quiet)

    # update kinematics only after the quaternion is initialized
    if state.mahony.is_initialized:
        sim.update_kinematics(state.kinematics, state.mahony.q, ax, ay, az, dt, is_zvw, state.zvw.dwell_counter)

        # The exact tick the dwell counter hits DWELL (20), the rollback finishes.
        # This means the foot has successfully completed a step and planted. 
        if state.zvw.dwell_counter == sim.DWELL:
            step_coords.append(np.copy(state.kinematics.position[:2]))

    # save position for plotting
    trajectory_x.append(state.kinematics.position[0])
    trajectory_y.append(state.kinematics.position[1])

print("Simulation Complete")

mapper.calculate_walls(step_coords)
    

Starting firmware simulation test: 
Simulation Complete
Calculated grid offset: -6.91 deg
Wall 1: 6.47 m
Wall 2: 2.40 m
Wall 3: 6.65 m
Wall 4: 2.37 m
Total calculated distance: 17.89 m
- - - - - - -
Wall: 1, Steps: 4, Calculated last step: 1.38 m -> (total: 6.47)
Wall: 2, Steps: 2, Calculated last step: 1.55 m -> (total: 2.40)
Wall: 3, Steps: 5, Calculated last step: 0.83 m -> (total: 6.65)
Wall: 4, Steps: 2, Calculated last step: 1.65 m -> (total: 2.37)


In [81]:
fig, ax = plt.subplots(figsize=(10,6))

ax.plot(trajectory_x, trajectory_y, color='tab:blue', label='Simulated streaming trajectory')
ax.scatter([trajectory_x[0]], [trajectory_y[0]], color='green', marker='*', s=100, label='Start', zorder=5)
ax.scatter([trajectory_x[-1]], [trajectory_y[-1]], color='red', marker='X', s=100, label='End', zorder=5)
ax.axis('equal')
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_title('Streaming Firmware Simulation Trajectory')
plt.legend()
plt.show()

gap = np.sqrt((trajectory_x[-1] - trajectory_x[0])**2 + (trajectory_y[-1] - trajectory_y[0])**2)
print(f"Streaming Raw Closure Gap: {gap:.3f} m")

Streaming Raw Closure Gap: 1.226 m


In [85]:
import os

# Select 1 straight line and 1 closed loop to act as our Oracles
logs_to_freeze = [
    '../data/measured_walks/initial_walk_test_10-08-2026_16-31-45_3.csv',
    '../data/measured_closed_loops/closed_loop_18-08-2026_20-44-45_3_CW_18m.csv',
    '../data/measured_closed_loops/closed_loop_18-08-2026_21-18-50_1_CW_NoStop_18m.csv',
    '../data/measured_closed_loops/closed_loop_18-08-2026_21-07-25_4_CCW_18m.csv',
    '../data/measured_closed_loops/closed_loop_18-08-2026_21-15-02_long_loop_CCW.csv'

]

save_path = '../data/golden_references/'
os.makedirs(save_path, exist_ok=True)

for filepath in logs_to_freeze:
    filename = os.path.basename(filepath)
    print(f"Generating Golden Reference for: {filename}")
    
    df = pd.read_csv(filepath, skipinitialspace=True)
    state = sim.SystemState()

    golden_stream = []
    step_coords = [[0.0, 0.0]] # The load-bearing origin anchor
    t_last = df['t_us'].iloc[0]

    for index, row in df.iterrows():
        ax, ay, az = row['ax'], row['ay'], row['az']
        gx, gy, gz = row['gx'], row['gy'], row['gz']
        t_current = row['t_us']

        dt = (t_current - t_last) / 1e6
        t_last = t_current
        if dt <= 0: 
            dt = 0.005

        # --- THE FROZEN ORACLE PIPELINE ---
        is_zvw = sim.update_zvw(state.zvw, ax, ay, az, gx, gy, gz)
        sim.update_mahony(state.mahony, ax, ay, az, gx, gy, gz, dt, state.zvw.instant_quiet)

        if state.mahony.is_initialized:
            sim.update_kinematics(state.kinematics, state.mahony.q, ax, ay, az, dt, is_zvw, state.zvw.dwell_counter)

            # Step extraction
            if state.zvw.dwell_counter == sim.DWELL:
                step_coords.append(np.copy(state.kinematics.position[:2]).tolist())

        # Save the exact internal state for every single tick
        golden_stream.append({
            'seq': row['seq'], # Keeping the original sequence number to perfectly align logs
            'is_zvw': int(is_zvw),
            'instant_quiet': int(state.zvw.instant_quiet),
            'qw': state.mahony.q[0], 'qx': state.mahony.q[1], 'qy': state.mahony.q[2], 'qz': state.mahony.q[3],
            'pos_x': state.kinematics.position[0], 'pos_y': state.kinematics.position[1], 'pos_z': state.kinematics.position[2]
        })

    # --- SAVE TO DISK ---
    base_name = filename.replace('.csv', '')
    
    # 1. The Continuous Stream (ZVW, Quaternions, Integration)
    df_stream = pd.DataFrame(golden_stream)
    stream_out = f"{save_path}{base_name}_golden_stream.csv"
    df_stream.to_csv(stream_out, index=False)
    
    # 2. The Extracted Steps (For Manhattan Snapping verification)
    df_steps = pd.DataFrame(step_coords, columns=['X', 'Y'])
    steps_out = f"{save_path}{base_name}_golden_steps.csv"
    df_steps.to_csv(steps_out, index=False)
    
    # Print the final golden position
    final_pos = state.kinematics.position
    print(f"  -> Saved {len(df_stream)} ticks and {len(df_steps)} steps.")
    print(f"  -> Final XYZ: [{final_pos[0]:.4f}, {final_pos[1]:.4f}, {final_pos[2]:.4f}]\n")

print("Stage 1 Officially Frozen. Oracles generated.")

Generating Golden Reference for: initial_walk_test_10-08-2026_16-31-45_3.csv
  -> Saved 6386 ticks and 15 steps.
  -> Final XYZ: [-19.8286, 0.0153, -0.0486]

Generating Golden Reference for: closed_loop_18-08-2026_20-44-45_3_CW_18m.csv
  -> Saved 10132 ticks and 14 steps.
  -> Final XYZ: [-0.0275, -0.0727, -0.0189]

Generating Golden Reference for: closed_loop_18-08-2026_21-18-50_1_CW_NoStop_18m.csv
  -> Saved 4587 ticks and 14 steps.
  -> Final XYZ: [-0.0171, -0.0774, -0.2818]

Generating Golden Reference for: closed_loop_18-08-2026_21-07-25_4_CCW_18m.csv
  -> Saved 9290 ticks and 13 steps.
  -> Final XYZ: [-0.7824, -0.8312, -0.5570]

Generating Golden Reference for: closed_loop_18-08-2026_21-15-02_long_loop_CCW.csv
  -> Saved 10681 ticks and 37 steps.
  -> Final XYZ: [0.4498, -0.6179, -2.7095]

Stage 1 Officially Frozen. Oracles generated.
